# Variables disponibles antes del partido

Reconstruimos forma, fuerza histórica, plantilla y mercado con estados temporales. Las dos ramas usarán los mismos partidos de comparación.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Construcción cronológica

Todos los partidos de una fecha leen el estado previo antes de actualizarlo con resultados del día. La implementación está en `mlcore/data.py`.

In [2]:
from mlcore.data import prepare
if not (OUT/'dataset.csv').exists(): prepare(OUT)
df, sets = read_data(OUT)
print('Cohorte:', df.shape, '| train/val:', int((~df.Season.isin(TEST)).sum()), '| test:', int(df.Season.isin(TEST).sum()))
display(pd.DataFrame({'grupo':sets.keys(), 'variables':[len(v) for v in sets.values()]}))

Cohorte: (36119, 290) | train/val: 30120 | test: 5999


/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


,grupo,variables
0,history,38
1,roster,6
2,skills,33
3,sin_cuotas,77
4,mercado,2
5,consenso,3
6,con_cuotas,84


**Qué encontramos / decisión.** Los partidos sin mercado quedan fuera de la comparación pareada pero ya alimentaron el historial. No se hace poda global por correlación.

## Disponibilidad y calidad

Se exige snapshot estrictamente anterior. Pinnacle se deja de usar como precio fiable desde el 23/07/2025.

In [3]:
assert ((df.H_snapshot_at < df.Date) | df.H_snapshot_at.isna()).all()
assert ((df.A_snapshot_at < df.Date) | df.A_snapshot_at.isna()).all()
assert not df.loc[df.Date.ge('2025-07-23'),'pinnacle_fiable'].any()
display(pd.read_csv(OUT/'coverage.csv'))
assert not any(c.startswith(('MKT','REF','PIN')) for c in sets['sin_cuotas'])

,Season,n,cobertura_plantilla,antiguedad_H,antiguedad_A,pinnacle_fiable
0,1415,2984,0.892091,125.653644,125.653644,0.994973
1,1516,2984,0.980228,154.235015,154.168360,0.998660
2,1617,2984,0.987265,155.279501,155.129555,0.992292
3,1718,2984,0.988606,154.249410,154.100775,0.987265
4,1819,2984,0.996649,150.744881,150.751259,0.994973
5,1920,2801,0.993217,169.424427,169.339305,0.997501
6,2021,3163,0.996522,141.559848,141.486538,0.998419
7,2122,3123,0.987192,166.511441,164.619401,0.999360
8,2223,3057,0.991168,179.760105,181.637648,0.999346
9,2324,3056,0.991819,155.681445,155.126233,0.999346


**Qué encontramos / decisión.** La cobertura inferior al 100% y los ratings antiguos se reportan. No se rellenan habilidades con una media de toda la temporada.

## Separación y diagnóstico de variables

La revisión descriptiva no elimina columnas silenciosamente; las decisiones aprendidas se ejecutarán dentro de los folds.

In [4]:
tv = df[~df.Season.isin(TEST)]
review = tv[sets['sin_cuotas']+sets['mercado']].agg(['count','mean','std']).T
review['missing_fraction'] = tv[review.index].isna().mean()
review.to_csv(OUT/'feature_diagnostics_trainval.csv')
display(review.head(15))
print('Orden cronologico:', df.Date.is_monotonic_increasing)

,count,mean,std,missing_fraction
H_GF_5,30120.0,1.380128,0.666959,0.000000
H_GA_5,30120.0,1.405626,0.616666,0.000000
H_Pts_5,30120.0,1.357502,0.683440,0.000000
H_CS_5,30120.0,0.264352,0.207209,0.000000
H_Shots_5,26641.0,12.133775,2.977067,0.115505
H_SOT_5,26641.0,4.406891,1.401765,0.115505
H_Corners_5,26641.0,4.920590,1.427086,0.115505
H_Fouls_5,26161.0,12.781125,2.556934,0.131441
H_Cards_5,26641.0,2.188002,0.788724,0.115505
H_HPPG,30120.0,1.404452,0.363176,0.000000


Orden cronologico: True


**Qué encontramos / decisión.** Se exportan train_val y test en el directorio de ejecución. La tabla es diagnóstica; la selección efectiva será trazable por el pipeline.